# Обтекание препятствия идеальной жидкостью

## Базовый уровень

Смоделируем ламинарный поток в прямом канале. Будем решать уравнение Лапласа на функцию тока $\psi$

$$\nabla^2 \psi = 0.$$

На верхней и нижней стенках канала функция тока постоянна ($\psi(x, 1) = 1$, $\psi(x, 0) = 0$), на входе и выходе -- линейно изменяется.

Компоненты скорости выражаеются через функцию тока как $u = \frac{\partial \psi}{\partial y}$, $v = -\frac{\partial \psi}{\partial x}$.

Будем решать уравнение Лапласа методом Гаусса-Зейделя.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
Lx = 4.0
Ly = 1.0
Nx = 240
Ny = 80
dx = Lx / (Nx - 1)
dy = Ly / (Ny - 1)
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
X, Y = np.meshgrid(x, y, indexing='xy')

U = 1.0 / Ly

max_iter = 20000
tol = 1e-6

In [ ]:
def solve_laplace(psi0, mask):
    psi = psi0.copy()
    Ny, Nx = psi.shape
    dx2 = dx*dx
    dy2 = dy*dy
    denom = 2*(dx2 + dy2)
    for it in range(max_iter):
        maxdiff = 0.0
        for j in range(1, Ny-1):
            for i in range(1, Nx-1):
                if not mask[j,i]:
                    continue
                new_val = ((psi[j, i+1] + psi[j, i-1]) * dy2 + (psi[j+1, i] + psi[j-1, i]) * dx2) / denom
                diff = abs(new_val - psi[j,i])
                if diff > maxdiff:
                    maxdiff = diff
                psi[j,i] = new_val
        if (it % 500 == 0) or (it == max_iter-1):
            print(f"Итерация {it}, max diff = {maxdiff:.3e}")
        if maxdiff < tol:
            print(f"Метод сошёлся за {it} итерации, max diff={maxdiff:.3e}")
            break
    return psi

In [ ]:
psi0 = np.zeros((Ny, Nx))
psi0[0, :] = 0.0
psi0[-1, :] = 1.0
for j in range(Ny):
    psi0[j, 0] = y[j] / Ly
    psi0[j, -1] = y[j] / Ly

mask = np.ones_like(psi0, dtype=bool)
mask[0, :] = False
mask[-1, :] = False
mask[:, 0] = False
mask[:, -1] = False

psi = solve_laplace(psi0, mask)

u = np.zeros_like(psi)
v = np.zeros_like(psi)
u[1:-1, :] = (psi[2:, :] - psi[:-2, :]) / (2*dy)
v[:, 1:-1] = -(psi[:, 2:] - psi[:, :-2]) / (2*dx)
u[0, :] = (psi[1, :] - psi[0, :]) / dy
u[-1, :] = (psi[-1, :] - psi[-2, :]) / dy
v[:, 0] = -(psi[:, 1] - psi[:, 0]) / dx
v[:, -1] = -(psi[:, -1] - psi[:, -2]) / dx

Визуализируем линии тока и поле скоростей.

In [ ]:
plt.figure(figsize=(10,4))
plt.streamplot(X, Y, u, v, color='blue')
plt.title("Линии тока для течения в канале")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.quiver(X[::8, ::24], Y[::8, ::24], u[::8, ::24], v[::8, ::24], color='blue', scale=22)
plt.title("Поле скорости для течения в канале")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

## Продвинутый уровень

Решим задачу об обтекании цилиндра. Для этого поместим в центр канала цилиндр. На границе цилиндра зададим $\psi = \text{const}$.

In [ ]:
a = 0.12
x0 = Lx * 0.45
y0 = Ly * 0.5

cyl_mask = (X - x0)**2 + (Y - y0)**2 <= a*a

psi_cyl_value = y0 / Ly

psi_c = psi0.copy()
psi_c[cyl_mask] = psi_cyl_value

mask_cyl = mask.copy()
mask_cyl[cyl_mask] = False

psi_with_cyl = solve_laplace(psi_c, mask_cyl)

u2 = np.zeros_like(psi_with_cyl)
v2 = np.zeros_like(psi_with_cyl)
u2[1:-1, :] = (psi_with_cyl[2:, :] - psi_with_cyl[:-2, :]) / (2*dy)
v2[:, 1:-1] = -(psi_with_cyl[:, 2:] - psi_with_cyl[:, :-2]) / (2*dx)
u2[0, :] = (psi_with_cyl[1, :] - psi_with_cyl[0, :]) / dy
u2[-1, :] = (psi_with_cyl[-1, :] - psi_with_cyl[-2, :]) / dy
v2[:, 0] = -(psi_with_cyl[:, 1] - psi_with_cyl[:, 0]) / dx
v2[:, -1] = -(psi_with_cyl[:, -1] - psi_with_cyl[:, -2]) / dx

In [ ]:
plt.figure(figsize=(10,4))
plt.streamplot(X, Y, u2, v2, color='blue', broken_streamlines=False)
theta = np.linspace(0, 2*np.pi, 200)
plt.plot(x0 + a*np.cos(theta), y0 + a*np.sin(theta), linewidth=2)
plt.title("Линии тока для обтекания цилиндра")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.quiver(X[::4, ::6], Y[::4, ::6], u2[::4, ::6], v2[::4, ::6], color='blue', scale=40)
theta = np.linspace(0, 2*np.pi, 200)
plt.plot(x0 + a*np.cos(theta), y0 + a*np.sin(theta), linewidth=2)
plt.title("Поле скорости для обтекания цилиндра")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

Построим график распределения скорости вдоль центральной линии канала и сравним его с аналитическим решением для обтекания цилиндра в бесконечном потоке.

In [ ]:
j0 = np.argmin(np.abs(y - y0))
psi_line = psi_with_cyl[j0, :]
u_line = u2[j0, :]
v_line = v2[j0, :]
velocity = np.sqrt(u_line**2 + v_line**2)

# аналитическое решение
x1 = x - x0
u_analytic = np.zeros_like(x1)
u_analytic[np.abs(x1) > a] = U*(1 - a*a / (x1[np.abs(x1) > a]**2))
velocity_analytic = np.abs(u_analytic)

plt.figure(figsize=(9,4))
plt.plot(x, velocity, label='численное')
plt.plot(x[np.abs(x1) > a], velocity_analytic[np.abs(x1) > a], '--', label='аналитическое')
# полосой отметим на графике цилиндр
plt.axvspan(x0 - a, x0 + a, alpha=0.2)
plt.xlabel("x")
plt.ylabel("скорость")
plt.legend()
plt.title("График распределения скорости вдоль центральной линии канала")
plt.grid(True)
plt.show()
